In [0]:
# ============================================================
# NOTEBOOK 01 — Bronze Ingestion
# Medallion Architecture: Raw CSV → Bronze Parquet
# ============================================================
# This notebook reads the three raw CSV files from the Unity
# Catalog volume and saves them as Parquet (Bronze layer).
# No transformations are applied here — Bronze is raw data only.
# ============================================================

# Define the path to our uploaded raw files in Unity Catalog
RAW_PATH = "/Volumes/workspace/default/flight_raw_data"

# Define where Bronze Parquet files will be saved
BRONZE_PATH = "/Volumes/workspace/default/flight_raw_data/bronze"

print("Raw data path:", RAW_PATH)
print("Bronze output path:", BRONZE_PATH)

Raw data path: /Volumes/workspace/default/flight_raw_data
Bronze output path: /Volumes/workspace/default/flight_raw_data/bronze


In [0]:
# ============================================================
# Step 1 — Read raw CSV files into PySpark DataFrames
# ============================================================
# spark is automatically available in Databricks notebooks.
# inferSchema=True tells Spark to detect column data types
# automatically by sampling the file.
# header=True tells Spark the first row contains column names.
# ============================================================

# Read flights — the main dataset (~5.8 million rows)
df_flights = spark.read.csv(
    f"{RAW_PATH}/flights.csv",
    header=True,
    inferSchema=True
)

# Read airlines — maps carrier codes to full airline names
df_airlines = spark.read.csv(
    f"{RAW_PATH}/airlines.csv",
    header=True,
    inferSchema=True
)

# Read airports — maps airport codes to city, state, location
df_airports = spark.read.csv(
    f"{RAW_PATH}/airports.csv",
    header=True,
    inferSchema=True
)

print("flights row count:", df_flights.count())
print("airlines row count:", df_airlines.count())
print("airports row count:", df_airports.count())

flights row count: 5819079
airlines row count: 14
airports row count: 322


In [0]:
# ============================================================
# Step 2 — Preview the data and print schemas
# ============================================================
# schema() shows column names and detected data types.
# show(5) prints the first 5 rows in a readable table format.
# ============================================================

print("=== FLIGHTS SCHEMA ===")
df_flights.printSchema()

print("=== FLIGHTS SAMPLE (5 rows) ===")
df_flights.show(5)

print("=== AIRLINES SAMPLE ===")
df_airlines.show(14)

print("=== AIRPORTS SAMPLE (5 rows) ===")
df_airports.show(5)

=== FLIGHTS SCHEMA ===
root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIV

In [0]:
# ============================================================
# Step 3 — Save all three DataFrames as Parquet (Bronze layer)
# ============================================================
# mode("overwrite") means if the file already exists, replace it.
# This makes the notebook safe to re-run multiple times without
# errors. Parquet is saved as a folder containing part files —
# this is normal Spark behaviour, not a single file.
# ============================================================

print("Saving flights to Bronze Parquet...")
df_flights.write.mode("overwrite").parquet(f"{BRONZE_PATH}/flights")

print("Saving airlines to Bronze Parquet...")
df_airlines.write.mode("overwrite").parquet(f"{BRONZE_PATH}/airlines")

print("Saving airports to Bronze Parquet...")
df_airports.write.mode("overwrite").parquet(f"{BRONZE_PATH}/airports")

print("Bronze ingestion complete.")
print(f"Bronze files saved to: {BRONZE_PATH}")

Saving flights to Bronze Parquet...
Saving airlines to Bronze Parquet...
Saving airports to Bronze Parquet...
Bronze ingestion complete.
Bronze files saved to: /Volumes/workspace/default/flight_raw_data/bronze
